# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates step-by-step loading, exploration, and processing of the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset entities (record sets, fields, columns) are referenced using their Croissant `@id` for clarity and reproducibility.

### Dataset Source
The dataset is provided as a [Croissant](https://mlcommons.org/croissant/) schema:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant URL (Croissant Schema JSON-LD)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print general metadata (as attributes, not as dict items)
meta = dataset.metadata
print(f"Dataset Name   : {meta.name}")
print(f"Description    : {meta.description}")
print(f"Published      : {meta.datePublished}")
print(f"License        : {meta.license}")

## 2. Data Overview
### List available record sets and their fields/columns by `@id`

In Croissant, data is partitioned into **record sets**. We'll enumerate all record sets and list their fields/columns using their Croissant `@id`.

In [ ]:
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in this dataset's metadata.")
else:
    for rs in record_sets:
        print(f"\nRecord Set: {rs.name} (@id: {rs.id})")
        if rs.fields:
            for field in rs.fields:
                print(f"  - Field: {getattr(field, 'name', '')} (@id: {field.id}), dtype: {getattr(field, 'data_type', '')}")
        elif hasattr(rs, 'columns') and rs.columns:
            for col in rs.columns:
                print(f"  - Column: {getattr(col, 'name', '')} (@id: {col.id}), dtype: {getattr(col, 'data_type', '')}")
        else:
            print("  [No fields or columns available in record set]")

### Preview a sample record from the main record set

Let's use the first available record set, and print a sample record to understand its structure. _All references will use the entity `@id`._

In [ ]:
# Get the first record set by @id
if not record_sets:
    raise Exception("No record sets found in metadata!")
main_record_set = record_sets[0]
main_record_set_id = main_record_set.id
print(f"Previewing records for record set: {main_record_set_id}")
for i, rec in enumerate(dataset.records(record_set=main_record_set_id)):
    print(rec)
    if i == 2:
        break

## 3. Data Extraction

We'll load **all available record sets** into Pandas DataFrames. Each DataFrame will be keyed by the record set `@id` as per the Croissant schema. Column names correspond to field or column `@id`s.

In [ ]:
# List of all record set @id's
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set {record_set_id}")
    else:
        print(f"No records loaded for record set {record_set_id}")

# Preview columns of the main record set (first one)
print("\nColumns for main record set:")
print(dataframes[main_record_set_id].columns.tolist())

dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

- Select a numeric field by its Croissant `@id` for filtering and normalization
- Filter, normalize, and group records for summary overview

> **Note:** We'll look for a common quantitative or numeric field (e.g., molecular weight, peptide counts, coverage percent) _by inspecting available columns above_. Adjust the variables as needed.

In [ ]:
# Select a numeric field for filtering and analysis
# Replace with the actual field @id seen above (e.g., 'cr:molecular_weight' or similar from your dataset)
numeric_field_id = None
for c in dataframes[main_record_set_id].columns:
    if 'weight' in c.lower() or 'mw' in c.lower() or 'peptide' in c.lower() or 'coverage' in c.lower():
        numeric_field_id = c
        break
if not numeric_field_id:
    numeric_field_id = dataframes[main_record_set_id].select_dtypes('number').columns[0]
print(f"Using numeric field {numeric_field_id} for analysis.")

# Ensure conversion to numeric (some Croissant data can be loaded as str by default)
df = dataframes[main_record_set_id].copy()
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter by threshold
threshold = df[numeric_field_id].quantile(0.75)
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold:.2f}")
print(filtered_df[[numeric_field_id]].head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field (e.g., description or accession)
group_field_id = None
for c in df.columns:
    if 'description' in c.lower() or 'accession' in c.lower():
        group_field_id = c
        break

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"Mean {numeric_field_id} grouped by {group_field_id} (first 5 groups):")
    print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the selected numeric field and its normalized values. We'll use `matplotlib` and `seaborn` for easy visualization.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

fig, axs = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True, ax=axs[0])
axs[0].set_title(f"Distribution of {numeric_field_id}")

sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=30, kde=True, ax=axs[1], color='orange')
axs[1].set_title(f"Normalized {numeric_field_id}\n(filtered records)")

plt.tight_layout()
plt.show()

## 6. Conclusion

- We used `mlcroissant` to programmatically explore the FAIR^2 dataset via its Croissant schema.
- All record sets, fields, and columns were referenced using their stable `@id` attributes.
- The workflow showcased how to load, filter, normalize, group, and visualize data for downstream analysis.

For more, consult [mlcroissant documentation](https://mlcommons.org/croissant/) or the dataset's Croissant schema for extended options.